In [97]:
# install importance library
!pip install -q opencv-python matplotlib scipy pandas tqdm pyyaml

In [98]:
!pip install -q -U ultralytics

In [81]:
import numpy as np
import pandas as pd
import os
import shutil
import torch

In [82]:
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name:    Tesla T4


In [85]:
os.chdir('/content')
!ls

Low-Light-Image-Enhancement-and-Downstream-Recognition	sample_data


In [84]:
if os.path.exists("Low-Light-Image-Enhancement-and-Downstream-Recognition"):
    shutil.rmtree("Low-Light-Image-Enhancement-and-Downstream-Recognition")

!git clone https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition

Cloning into 'Low-Light-Image-Enhancement-and-Downstream-Recognition'...
remote: Enumerating objects: 14779, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 14779 (delta 6), reused 18 (delta 3), pack-reused 14747 (from 1)
Receiving objects: 100% (14779/14779), 269.17 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (14713/14713), done.


In [86]:
os.chdir('/content/Low-Light-Image-Enhancement-and-Downstream-Recognition')
%ls

Dataset/  Doc/  Notebooks/  README.md  requirements.txt  run.py  src/


# Chose model to run

In [87]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

In [88]:
%cd "/content/Low-Light-Image-Enhancement-and-Downstream-Recognition/Dataset/exdark_yolo_dark/"
%ls

/content/Low-Light-Image-Enhancement-and-Downstream-Recognition/Dataset/exdark_yolo_dark
data.yaml  README.dataset.txt  README.roboflow.txt  test/  train/  valid/


# Module Edge Loss use pytorch

In [117]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO

# 1. Kiểm tra phần cứng theo chuẩn slide bài giảng
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[*] Thiết bị sử dụng: {device}")

# 2. Định nghĩa Module Edge Loss khả vi bằng toán tử Sobel (Slide Image Processing) [source: 2]
class SobelEdgeLoss(nn.Module):
    def __init__(self, device='cuda'):
        super(SobelEdgeLoss, self).__init__()
        # Kernel Sobel đạo hàm cấp 1 theo phương ngang và dọc [source: 2]
        sobel_x = torch.tensor([[-1., 0., 1.],
                                [-2., 0., 2.],
                                [-1., 0., 1.]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        sobel_y = torch.tensor([[-1., -2., -1.],
                                [ 0.,  0.,  0.],
                                [ 1.,  2.,  1.]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer('sobel_x', sobel_x.to(device))
        self.register_buffer('sobel_y', sobel_y.to(device))
        self.loss_fn = nn.L1Loss()

    def get_edges(self, x):
        if x.shape[1] > 1:
            x = torch.mean(x, dim=1, keepdim=True)
        grad_x = F.conv2d(x, self.sobel_x, padding=1)
        grad_y = F.conv2d(x, self.sobel_y, padding=1)
        return torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-6)

    def forward(self, feature_map, input_img):
        pred_edge = self.get_edges(feature_map)
        if pred_edge.shape[-2:] != input_img.shape[-2:]:
            input_img = F.interpolate(input_img, size=pred_edge.shape[-2:], mode='bilinear', align_corners=False)
        target_edge = self.get_edges(input_img)
        return self.loss_fn(pred_edge, target_edge)

# 3. Khởi tạo biến lưu trữ toàn cục và mô hình YOLOv8
shallow_features_storage = {}
edge_loss_module = SobelEdgeLoss(device=device)

# Load checkpoint pretrained
model = YOLO('yolov8n.pt')

# Hàm Hook trích xuất đặc trưng tầng nông của Backbone
def hook_shallow_layer(module, input, output):
    shallow_features_storage['feat'] = output

# model.model.model[1] là tầng Conv đầu tiên trong Backbone YOLOv8
model.model.model[1].register_forward_hook(hook_shallow_layer)

# 4. Định nghĩa Callback nhúng Edge Loss vào vòng lặp tối ưu Gradient Descent [source: 9, 11]
def on_train_batch_end_callback(trainer):
    if 'feat' in shallow_features_storage:
        # Chuẩn hóa ảnh batch về khoảng [0, 1] [source: 9]
        imgs = trainer.batch['img'].to(trainer.device).float() / 255.0
        edge_loss = edge_loss_module(shallow_features_storage['feat'], imgs)

        # Hệ số phạt biên cạnh (tránh làm lệch hàm loss phát hiện chính)
        lambda_edge = 0.05
        # Lan truyền ngược gradient cho hàm loss bổ sung [source: 9]
        (lambda_edge * edge_loss).backward()

# Xóa callback cũ (nếu có) và thêm callback mới
model.callbacks["on_train_batch_end"] = [on_train_batch_end_callback]

# 5. Kích hoạt huấn luyện
YAML_PATH = "/content/Low-Light-Image-Enhancement-and-Downstream-Recognition/Dataset/exdark_yolo_dark/data.yaml"

results = model.train(
    data=YAML_PATH,
    epochs=30,
    batch=8,
    imgsz=640,
    device=0 if device == 'cuda' else 'cpu'
)

[*] Thiết bị sử dụng: cuda
Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Low-Light-Image-Enhancement-and-Downstream-Recognition/Dataset/exdark_yolo_dark/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937